In [ ]:
!pip install regex datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import regex as re
import numpy as np
import pickle

pat = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

def merge(ids, pair, idx):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def encode(text, merges, cache):
    out = []
    for w in pat.findall(text):
        if w not in cache:
            ids = list(w.encode("utf-8"))
            while len(ids) >= 2:
                pairs = set(zip(ids, ids[1:]))
                pair = min(
                    pairs,
                    key=lambda p: merges.get(p, float("inf"))
                )
                if pair not in merges:
                    break
                ids = merge(ids, pair, merges[pair])
            cache[w] = ids
        out.extend(cache[w])
    return out

def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/GPT_boga"

merges = pickle.load(
    open(f"{DRIVE_DIR}/merges.pkl", "rb")
)

EOT = 4096
cache = {}

## Building the dataset (train, val), all the experiments below were done before this

In [ ]:
import os

BLOCK_SIZE = 256

FT_DIR = f"{DRIVE_DIR}/finetuning"
os.makedirs(FT_DIR, exist_ok=True)

META_PREFIXES = ("Features:", "Words:", "Summary:", "Random sentences:")


In [ ]:
def iter_instruct_examples(split):

    ds = load_dataset(
        "roneneldan/TinyStoriesInstruct",
        split=split,
        streaming=True
    )

    current = []

    for ex in ds:

        line = ex["text"]
        current.append(line)

        if "<|endoftext|>" in line:

            yield "\n".join(current).strip()
            current = []

In [ ]:
def parse_example(text):

    lines = text.splitlines()

    metadata = []
    story_lines = []

    inside_story = False

    for line in lines:

        line = line.strip()

        if not line:
            continue

        if line == "<|endoftext|>":
            continue

        if line.startswith("Story:"):

            inside_story = True

            rest = line[len("Story:"):].strip()

            if rest:
                story_lines.append(rest)

            continue

        if line.startswith(META_PREFIXES):

            metadata.append(line)

            # Metadata appearing after Story marks
            # the end of the story body.
            inside_story = False

            continue

        if inside_story:
            story_lines.append(line)

    if not metadata or not story_lines:
        return None

    prompt = "\n".join(metadata) + "\nStory:\n"
    story = "\n".join(story_lines)

    return prompt, story


In [ ]:
def prepare_example(text):

    parsed = parse_example(text)

    if parsed is None:
        return None

    prompt, story = parsed

    prompt_ids = encode(prompt, merges, cache)
    story_ids = encode(story, merges, cache)

    ids = prompt_ids + story_ids + [EOT]

    # Only complete examples that fit the original
    # pretrained 256-token context.
    if len(ids) > BLOCK_SIZE:
        return None

    mask = (
        [0] * len(prompt_ids)
        + [1] * (len(story_ids) + 1)
    )

    assert len(ids) == len(mask)

    return ids, mask

In [ ]:
def build_sft_split(
    split,
    prefix,
    target_loss_tokens
):

    ids_path = f"{FT_DIR}/{prefix}_ids.bin"
    mask_path = f"{FT_DIR}/{prefix}_mask.bin"
    offsets_path = f"{FT_DIR}/{prefix}_offsets.npy"

    offsets = [0]

    total_tokens = 0
    total_loss_tokens = 0
    total_examples = 0

    seen_examples = 0
    too_long = 0
    invalid = 0

    with open(ids_path, "wb") as f_ids, \
         open(mask_path, "wb") as f_mask:

        for text in iter_instruct_examples(split):

            seen_examples += 1

            result = prepare_example(text)

            if result is None:

                # optional distinction for statistics
                parsed = parse_example(text)

                if parsed is None:
                    invalid += 1
                else:
                    prompt, story = parsed

                    n = (
                        len(encode(prompt, merges, cache))
                        + len(encode(story, merges, cache))
                        + 1
                    )

                    if n > BLOCK_SIZE:
                        too_long += 1

                continue

            ids, mask = result

            loss_tokens = sum(mask)

            # Don't overshoot the target by another example
            # once we've reached the requested budget.
            if (
                total_loss_tokens >= target_loss_tokens
            ):
                break

            np.asarray(
                ids,
                dtype=np.uint16
            ).tofile(f_ids)

            np.asarray(
                mask,
                dtype=np.uint8
            ).tofile(f_mask)

            total_tokens += len(ids)
            total_loss_tokens += loss_tokens
            total_examples += 1

            offsets.append(total_tokens)

    offsets = np.asarray(
        offsets,
        dtype=np.int64
    )

    np.save(offsets_path, offsets)

    print("=" * 60)
    print("split              :", split)
    print("examples seen      :", f"{seen_examples:,}")
    print("examples kept      :", f"{total_examples:,}")
    print("too long (>256)    :", f"{too_long:,}")
    print("invalid             :", f"{invalid:,}")
    print("total tokens        :", f"{total_tokens:,}")
    print("loss/story tokens   :", f"{total_loss_tokens:,}")
    print("average seq length  :", total_tokens / total_examples)
    print("average loss tokens :", total_loss_tokens / total_examples)

    print("\nFiles:")
    print(ids_path)
    print(mask_path)
    print(offsets_path)

In [ ]:
from datasets import load_dataset
import numpy as np

In [ ]:
build_sft_split(
    split="train",
    prefix="train",
    target_loss_tokens=8_000_000
)

split              : train
examples seen      : 89,191
examples kept      : 49,020
too long (>256)    : 40,170
invalid             : 0
total tokens        : 10,617,686
loss/story tokens   : 8,000,035
average seq length  : 216.59906160750714
average loss tokens : 163.19940840473276

Files:
/content/drive/MyDrive/GPT_boga/finetuning/train_ids.bin
/content/drive/MyDrive/GPT_boga/finetuning/train_mask.bin
/content/drive/MyDrive/GPT_boga/finetuning/train_offsets.npy


In [ ]:
build_sft_split(
    split="validation",
    prefix="val",
    target_loss_tokens=500_000
)

split              : validation
examples seen      : 5,027
examples kept      : 3,072
too long (>256)    : 1,953
invalid             : 1
total tokens        : 663,197
loss/story tokens   : 500,031
average seq length  : 215.88444010416666
average loss tokens : 162.7705078125

Files:
/content/drive/MyDrive/GPT_boga/finetuning/val_ids.bin
/content/drive/MyDrive/GPT_boga/finetuning/val_mask.bin
/content/drive/MyDrive/GPT_boga/finetuning/val_offsets.npy


## Prev experiments

In [ ]:
from datasets import load_dataset
import numpy as np

ds = load_dataset(
    "roneneldan/TinyStoriesInstruct",
    split="train",
    streaming=True
)

cache = {}

total_lens = []
story_lens = []
n = 5000   # enough for a good estimate

for i, ex in enumerate(ds):

    text = ex["text"]

    # total tokens
    total_ids = encode(text, merges, cache)
    total_lens.append(len(total_ids))

    # response/story-only tokens
    marker = "Story:"
    if marker in text:
        story_text = text.split(marker, 1)[1]
        story_ids = encode(story_text, merges, cache)
        story_lens.append(len(story_ids))

    if i + 1 >= n:
        break

total_lens = np.array(total_lens)
story_lens = np.array(story_lens)

print("Examples inspected:", len(total_lens))

print("\nTOTAL TOKENS PER EXAMPLE")
print("mean   :", total_lens.mean())
print("median :", np.median(total_lens))
print("p90    :", np.percentile(total_lens, 90))
print("p95    :", np.percentile(total_lens, 95))
print("max    :", total_lens.max())

print("\nSTORY / TARGET TOKENS PER EXAMPLE")
print("mean   :", story_lens.mean())
print("median :", np.median(story_lens))
print("p90    :", np.percentile(story_lens, 90))
print("p95    :", np.percentile(story_lens, 95))
print("max    :", story_lens.max())

for budget in [5_000_000, 10_000_000, 20_000_000]:
    approx_examples = budget / total_lens.mean()
    print(
        f"\n{budget/1e6:.0f}M total tokens "
        f"≈ {approx_examples:,.0f} examples"
    )

Examples inspected: 5000

TOTAL TOKENS PER EXAMPLE
mean   : 32.0482
median : 29.0
p90    : 61.0
p95    : 70.0
max    : 279

STORY / TARGET TOKENS PER EXAMPLE
mean   : 21.127450980392158
median : 1.0
p90    : 58.0
p95    : 65.0
max    : 143

5M total tokens ≈ 156,015 examples

10M total tokens ≈ 312,030 examples

20M total tokens ≈ 624,060 examples


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "roneneldan/TinyStoriesInstruct",
    split="train",
    streaming=True
)

for i, ex in enumerate(ds):
    print("=" * 80)
    print("EXAMPLE", i)
    print(repr(ex["text"]))
    print()

    if i == 9:
        break

EXAMPLE 0
'Features: Dialogue'

EXAMPLE 1
'Words: quit, oak, gloomy'

EXAMPLE 2
'Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.'

EXAMPLE 3
'Story: '

EXAMPLE 4
''

EXAMPLE 5
'Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sang songs.'

EXAMPLE 6
'But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go home and have some hot cocoa.'

EXAMPLE 7
'"Ben, I want to quit," she said. "It\'s too cold and dark. Let\'s go home."'

EXAMPLE 8
'Ben looked at Sara and frowned. He liked the oak tree and the park. He wanted to stay and play.'

EXAMPLE 9
'"No, Sara, don\'t quit," he said. "It\'s fun here. Look, there\'s a squirrel. Let\'s chase it."'



In [ ]:
for i, ex in enumerate(ds):

    text = ex["text"]

    if "Story:" in text:
        before, story = text.split("Story:", 1)

        print("=" * 80)
        print("FULL:")
        print(repr(text))

        print("\nPROMPT PART:")
        print(repr(before + "Story:"))

        print("\nSTORY PART:")
        print(repr(story))

        print("\nTOKEN COUNTS:")
        print("full :", len(encode(text, merges, cache)))
        print("story:", len(encode(story, merges, cache)))

    if i == 9:
        break

FULL:
'Story: '

PROMPT PART:
'Story:'

STORY PART:
' '

TOKEN COUNTS:
full : 4
story: 1


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "roneneldan/TinyStoriesInstruct",
    split="train",
    streaming=True
)

def iter_instruct_examples(ds):

    current = []

    for ex in ds:
        line = ex["text"]

        # A new example begins at Features:
        if line.startswith("Features:") and current:
            yield "\n".join(current).strip()
            current = []

        current.append(line)

    # final example
    if current:
        yield "\n".join(current).strip()

In [ ]:
for i, text in enumerate(iter_instruct_examples(ds)):

    print("=" * 100)
    print(f"EXAMPLE {i}")
    print(text)

    if i == 4:
        break

EXAMPLE 0
Features: Dialogue
Words: quit, oak, gloomy
Summary: Sara and Ben were playing in the park, but Sara wanted to go home because it was cold and dark. Ben convinced her to stay and play, but eventually agreed to go home and have hot cocoa.
Story: 

Sara and Ben were playing in the park. They liked to climb the big oak tree and pretend they were birds. They made nests with leaves and twigs and sang songs.
But today, the sky was gloomy and the wind was cold. Sara felt sad and cold. She wanted to go home and have some hot cocoa.
"Ben, I want to quit," she said. "It's too cold and dark. Let's go home."
Ben looked at Sara and frowned. He liked the oak tree and the park. He wanted to stay and play.
"No, Sara, don't quit," he said. "It's fun here. Look, there's a squirrel. Let's chase it."
Sara shook her head. She didn't want to chase the squirrel. She wanted to go home and have some hot cocoa.
"Please, Ben, let's go home," she said. "We can play here another day. I'm cold and hungry.

In [ ]:
import numpy as np

ds = load_dataset(
    "roneneldan/TinyStoriesInstruct",
    split="train",
    streaming=True
)

cache = {}

total_lens = []
prompt_lens = []
story_lens = []

n = 5000

for i, text in enumerate(iter_instruct_examples(ds)):

    # Full instruction + story
    full_ids = encode(text, merges, cache)

    if "Story:" not in text:
        continue

    prompt, story = text.split("Story:", 1)

    # Keep Story: as part of the prompt
    prompt = prompt + "Story:"

    prompt_ids = encode(prompt, merges, cache)
    story_ids = encode(story, merges, cache)

    total_lens.append(len(full_ids))
    prompt_lens.append(len(prompt_ids))
    story_lens.append(len(story_ids))

    if len(total_lens) >= n:
        break


total_lens = np.array(total_lens)
prompt_lens = np.array(prompt_lens)
story_lens = np.array(story_lens)

print("Examples inspected:", len(total_lens))

for name, arr in [
    ("TOTAL", total_lens),
    ("PROMPT", prompt_lens),
    ("STORY", story_lens),
]:

    print(f"\n{name} TOKENS")
    print("mean   :", arr.mean())
    print("median :", np.median(arr))
    print("p90    :", np.percentile(arr, 90))
    print("p95    :", np.percentile(arr, 95))
    print("p99    :", np.percentile(arr, 99))
    print("max    :", arr.max())

Examples inspected: 5000

TOTAL TOKENS
mean   : 501.5326
median : 346.0
p90    : 821.1000000000004
p95    : 1040.1000000000004
p99    : 1540.0700000000015
max    : 145885

PROMPT TOKENS
mean   : 39.0262
median : 27.0
p90    : 78.0
p95    : 88.0
p99    : 109.0
max    : 160

STORY TOKENS
mean   : 462.5064
median : 306.0
p90    : 775.0
p95    : 1006.0500000000002
p99    : 1480.1600000000035
max    : 145816


In [ ]:
block_size = 256
print("Fraction <= 256:",
      np.mean(total_lens <= 256))

print("Fraction <= 384:",
      np.mean(total_lens <= 384))

print("Fraction <= 512:",
      np.mean(total_lens <= 512))

Fraction <= 256: 0.204
Fraction <= 384: 0.5554
Fraction <= 512: 0.7062


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "roneneldan/TinyStoriesInstruct",
    split="train",
    streaming=True
)

def iter_instruct_examples(ds):

    current = []

    for ex in ds:

        line = ex["text"]
        current.append(line)

        # End of one complete instruction/story example
        if "<|endoftext|>" in line:

            text = "\n".join(current).strip()

            yield text

            current = []

In [ ]:
import numpy as np

cache = {}

total_lens = []
prompt_lens = []
story_lens = []

for text in iter_instruct_examples(ds):

    if "Story:" not in text:
        continue

    prompt, story = text.split("Story:", 1)

    prompt = prompt + "Story:"

    full_ids = encode(text, merges, cache)
    prompt_ids = encode(prompt, merges, cache)
    story_ids = encode(story, merges, cache)

    total_lens.append(len(full_ids))
    prompt_lens.append(len(prompt_ids))
    story_lens.append(len(story_ids))

    if len(total_lens) >= 5000:
        break

total_lens = np.array(total_lens)
prompt_lens = np.array(prompt_lens)
story_lens = np.array(story_lens)

for name, arr in [
    ("TOTAL", total_lens),
    ("PROMPT", prompt_lens),
    ("STORY", story_lens),
]:

    print(f"\n{name}")
    print("mean   :", arr.mean())
    print("median :", np.median(arr))
    print("p90    :", np.percentile(arr, 90))
    print("p95    :", np.percentile(arr, 95))
    print("p99    :", np.percentile(arr, 99))
    print("max    :", arr.max())

print("\nFraction <= 256:", np.mean(total_lens <= 256))
print("Fraction <= 384:", np.mean(total_lens <= 384))
print("Fraction <= 512:", np.mean(total_lens <= 512))


TOTAL
mean   : 276.402
median : 260.0
p90    : 352.10000000000036
p95    : 419.0500000000002
p99    : 640.0100000000002
max    : 1163

PROMPT
mean   : 53.7464
median : 55.0
p90    : 87.0
p95    : 95.05000000000018
p99    : 114.0
max    : 151

STORY
mean   : 222.6556
median : 206.0
p90    : 295.0
p95    : 351.0500000000002
p99    : 574.0300000000007
max    : 1075

Fraction <= 256: 0.4692
Fraction <= 384: 0.931
Fraction <= 512: 0.975


In [ ]:
from datasets import load_dataset

BLOCK_SIZE = 256

def iter_instruct_examples(split):

    ds = load_dataset(
        "roneneldan/TinyStoriesInstruct",
        split=split,
        streaming=True
    )

    current = []

    for ex in ds:

        line = ex["text"]
        current.append(line)

        # one complete record ends here
        if "<|endoftext|>" in line:

            text = "\n".join(current).strip()

            yield text

            current = []

In [ ]:
def prepare_example(text):

    if "Story:" not in text:
        return None

    prompt, story = text.split("Story:", 1)

    prompt = prompt + "Story:"

    # remove textual EOT because we'll append token 4096 ourselves
    story = story.replace("<|endoftext|>", "").strip()

    prompt_ids = encode(prompt, merges, cache)
    story_ids = encode(story, merges, cache)

    # append actual EOT token
    input_ids = prompt_ids + story_ids + [EOT]

    # only learn on story + EOT
    loss_mask = (
        [0] * len(prompt_ids)
        + [1] * (len(story_ids) + 1)
    )

    # keep only complete examples fitting pretrained context
    if len(input_ids) > BLOCK_SIZE:
        return None

    return input_ids, loss_mask

In [ ]:
vocab = {i: bytes([i]) for i in range(256)}

for (a, b), idx in merges.items():
    vocab[idx] = vocab[a] + vocab[b]

In [ ]:
count = 0

for text in iter_instruct_examples("train"):

    result = prepare_example(text)

    if result is None:
        continue

    input_ids, loss_mask = result

    print("=" * 80)
    print("tokens:", len(input_ids))
    print("loss tokens:", sum(loss_mask))
    print("decoded:")
    print(
        b"".join(
            (vocab[i] if i != EOT else b"<|endoftext|>")
            for i in input_ids
        ).decode("utf-8", errors="replace")
    )

    count += 1

    if count == 3:
        break

tokens: 221
loss tokens: 185
decoded:
Words: clean, neighbor, smooth
Summary: Lily and Sam clean Sam's muddy car together and become good friends.
Story:
Lily likes to clean her room. She puts her toys in the box, her books on the shelf, and her clothes in the drawer. She feels happy when her room is clean and smooth.
One day, Lily sees her neighbor Sam outside. Sam is playing with his car in the dirt. His car is brown and muddy. Lily thinks Sam's car needs to be clean and smooth too.
She goes outside and says hello to Sam. She asks him if he wants to clean his car with her. Sam says yes. They go to Lily's house and get a bucket of water and a sponge. They wash Sam's car until it is shiny and smooth.
Sam and Lily are proud of their work. They smile and thank each other. They decide to play with their cars on the sidewalk, where it is clean and smooth. They have fun and are good friends.<|endoftext|>
tokens: 240
loss tokens: 183
decoded:
Words: applaud, snake, good
Features: Dialogue
Su

In [ ]:
def prepare_example(text):

    parsed = parse_example(text)

    if parsed is None:
        return None

    prompt, story = parsed

    prompt_ids = encode(prompt, merges, cache)
    story_ids = encode(story, merges, cache)

    input_ids = prompt_ids + story_ids + [EOT]

    if len(input_ids) > BLOCK_SIZE:
        return None

    loss_mask = (
        [0] * len(prompt_ids)
        + [1] * (len(story_ids) + 1)
    )

    return input_ids, loss_mask

In [ ]:
META_PREFIXES = (
    "Features:",
    "Words:",
    "Summary:",
    "Random sentence:",
)

def parse_example(text):

    lines = text.splitlines()

    metadata = []
    story_lines = []

    inside_story = False

    for line in lines:

        line = line.strip()

        if not line:
            continue

        if line == "<|endoftext|>":
            continue

        if line.startswith("Story:"):

            inside_story = True

            # handle text appearing on same line after Story:
            rest = line[len("Story:"):].strip()

            if rest:
                story_lines.append(rest)

            continue

        if line.startswith(META_PREFIXES):

            metadata.append(line)

            # if metadata occurs after the story,
            # it marks the end of story text
            inside_story = False

            continue

        if inside_story:
            story_lines.append(line)

    if not story_lines:
        return None

    prompt = "\n".join(metadata) + "\nStory:\n"
    story = "\n".join(story_lines)

    return prompt, story